# BlinkMimic T4 Max 双模型对比评测

旧模型 `20261122_083958` 与新模型 `20261122_105913`。每次只让每张 T4 运行一个模型：GPU 0 评测旧模型，GPU 1 评测新模型；随后并行录制两个模型的视频。

使用 8 核 CPU 时每个进程 4 线程。结果分别写入两个 run 目录和 HF 私有仓库，不覆盖 checkpoint。

In [ ]:
from pathlib import Path
import os

RUNS = {
    'old_083958': {'run_id': '20261122_083958', 'gpu': '0'},
    'new_105913': {'run_id': '20261122_105913', 'gpu': '1'},
}
REPO_DIR = Path('/kaggle/working/proj2-humanoid-mimic/third_party/blinkmimic')
REPO_URL = 'https://github.com/fuyichen0725/blinkmimic.git'
COMMIT = 'd264658d6c9ce61fb52f9f0f8985a369f3ccc409'
BASE_RESULTS = Path('/kaggle/working/proj2-humanoid-mimic/results/blinkmimic_t4_ddp')
HF_REPO_NAME = 'proj2-blinkmimic-t4'
CPU_COUNT = len(os.sched_getaffinity(0))
CPU_THREADS = max(1, CPU_COUNT // len(RUNS))
VIDEO_SECONDS = 10
VIDEO_WIDTH = 640
VIDEO_HEIGHT = 360
VIDEO_FPS = 15
HF_TOKEN = os.environ.get('HF_TOKEN', '')
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN') or HF_TOKEN
except Exception as exc:
    print(f'kaggle_secrets unavailable: {exc}')
for item in RUNS.values():
    item['root'] = BASE_RESULTS / f"run_{item['run_id']}"
print('HF_TOKEN=' + ('set' if HF_TOKEN else 'MISSING'))
print({name: {'run_id': item['run_id'], 'gpu': item['gpu']} for name, item in RUNS.items()})


In [ ]:
import subprocess
print(subprocess.check_output(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'], text=True))
print(f'cpu_count={CPU_COUNT} threads_per_gpu={CPU_THREADS}')


In [ ]:
import subprocess
def run(command, cwd=None, check=True):
    print('$', ' '.join(str(x) for x in command))
    return subprocess.run(command, cwd=cwd, check=check, text=True)
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not (REPO_DIR / '.git').exists():
    run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO_DIR)])
run(['git', 'fetch', '--depth', '1', 'origin', COMMIT], cwd=REPO_DIR)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
if head != COMMIT:
    run(['git', 'reset', '--hard', COMMIT], cwd=REPO_DIR)
run(['uv', 'sync', '--frozen'], cwd=REPO_DIR)
motion_dir = REPO_DIR / 'data' / 'lafan1_g1'
motion_dir.mkdir(parents=True, exist_ok=True)
if len(list(motion_dir.glob('*.csv'))) < 8:
    run(['bash', 'tools/download_data.sh'], cwd=REPO_DIR)
csvs = sorted(motion_dir.glob('*.csv'))
print(f'csv_count={len(csvs)}')
if len(csvs) < 8:
    raise RuntimeError('动作数据不足 8 段')


In [ ]:
import shutil
import subprocess
import sys
try:
    from huggingface_hub import HfApi, hf_hub_download
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
    from huggingface_hub import HfApi, hf_hub_download
if not HF_TOKEN:
    raise RuntimeError('缺少 HF_TOKEN')
api = HfApi(token=HF_TOKEN)
hf_user = api.whoami()['name']
HF_REPO = f'{hf_user}/{HF_REPO_NAME}'
for name, item in RUNS.items():
    root = item['root']
    root.mkdir(parents=True, exist_ok=True)
    for filename in ('args.json', 'last.pt'):
        target = root / filename
        if not target.exists():
            remote = f"runs/run_{item['run_id']}/{filename}"
            path = hf_hub_download(repo_id=HF_REPO, filename=remote, local_dir=str(root / 'hf_download'), token=HF_TOKEN)
            shutil.copy(path, target)
    print(f"{name}: {root / 'last.pt'} {root.joinpath('last.pt').stat().st_size / 1e6:.1f} MB")


In [ ]:
import json
import time
from concurrent.futures import ThreadPoolExecutor
def launch_metrics(name, item):
    root = item['root']
    ckpt = str(root / 'last.pt')
    out = str(root / 'eval_final')
    command = ['uv', 'run', 'python', 'tools/eval_metrics.py', ckpt, '--out', out]
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = item['gpu']
    for key in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
        env[key] = str(CPU_THREADS)
    start = time.perf_counter()
    result = subprocess.run(command, cwd=REPO_DIR, env=env, text=True, capture_output=True)
    (root / 'eval_metrics_stdout.txt').write_text(result.stdout + '\n' + result.stderr)
    return name, result, time.perf_counter() - start
print('metrics: GPU0=old_083958, GPU1=new_105913, 4 CPU threads each')
with ThreadPoolExecutor(max_workers=2) as pool:
    results = list(pool.map(lambda pair: launch_metrics(*pair), RUNS.items()))
for name, result, seconds in results:
    print(f'{name}: exit={result.returncode} seconds={seconds:.1f}')
    if result.returncode != 0:
        print(result.stderr[-5000:])
        raise RuntimeError(f'{name} metrics failed')
    root = RUNS[name]['root']
    key = str(root / 'last.pt')
    meta = json.loads((root / 'eval_final.json').read_text())
    print(json.dumps({name: meta[key]}, indent=1))


In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor
def launch_record(name, item):
    root = item['root']
    command = ['uv', 'run', 'python', '-m', 'blinkmimic.evaluate', '--run_dir', str(root), '--ckpt', str(root / 'last.pt'), '--motion_dir', 'data/lafan1_g1', '--record', str(root / 'eval_traj.npz')]
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = item['gpu']
    for key in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
        env[key] = str(CPU_THREADS)
    start = time.perf_counter()
    result = subprocess.run(command, cwd=REPO_DIR, env=env, text=True, capture_output=True)
    (root / 'eval_record_stdout.txt').write_text(result.stdout + '\n' + result.stderr)
    return name, result, time.perf_counter() - start
with ThreadPoolExecutor(max_workers=2) as pool:
    results = list(pool.map(lambda pair: launch_record(*pair), RUNS.items()))
for name, result, seconds in results:
    print(f'{name}: exit={result.returncode} seconds={seconds:.1f}')
    if result.returncode != 0:
        print(result.stderr[-5000:])
        raise RuntimeError(f'{name} record failed')


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path
if shutil.which('ffmpeg') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'imageio-ffmpeg'], check=True)
    import imageio_ffmpeg
    bindir = Path('/kaggle/working/bin')
    bindir.mkdir(exist_ok=True)
    link = bindir / 'ffmpeg'
    if not link.exists(): link.symlink_to(imageio_ffmpeg.get_ffmpeg_exe())
    os.environ['PATH'] = str(bindir) + ':' + os.environ['PATH']
names = [p.stem for p in sorted((REPO_DIR / 'data' / 'lafan1_g1').glob('*.csv'))]
def render_one(name, item):
    root = item['root']
    clips = names[:4]
    command = ['uv', 'run', 'python', 'tools/render.py', str(root / 'eval_traj.npz'), str(root / 'eval_video.mp4'), '--clips', *clips, '--seconds', str(VIDEO_SECONDS), '--fps', str(VIDEO_FPS), '--width', str(VIDEO_WIDTH), '--height', str(VIDEO_HEIGHT), '--title', f"BlinkMimic-T4 {item['run_id']}"]
    env = os.environ.copy()
    env['MUJOCO_GL'] = 'egl'
    env['CUDA_VISIBLE_DEVICES'] = item['gpu']
    result = subprocess.run(command, cwd=REPO_DIR, env=env, text=True, capture_output=True)
    (root / 'eval_render_stdout.txt').write_text(result.stdout + '\n' + result.stderr)
    return name, result
with ThreadPoolExecutor(max_workers=2) as pool:
    results = list(pool.map(lambda pair: render_one(*pair), RUNS.items()))
for name, result in results:
    print(f'{name}: video_exit={result.returncode}')
    if result.returncode != 0: print(result.stderr[-4000:])


In [ ]:
from IPython.display import Video, display
if HF_TOKEN:
    for name, item in RUNS.items():
        root = item['root']
        for path in [root / 'eval_final.md', root / 'eval_final.json', root / 'eval_traj.npz', root / 'eval_video.mp4', root / 'eval_metrics_stdout.txt', root / 'eval_record_stdout.txt', root / 'eval_render_stdout.txt']:
            if path.exists():
                api.upload_file(path_or_fileobj=str(path), path_in_repo=f"runs/run_{item['run_id']}/eval/{path.name}", repo_id=HF_REPO, repo_type='model')
                print(f'[hf] uploaded {name}/{path.name}')
for name, item in RUNS.items():
    root = item['root']
    key = str(root / 'last.pt')
    meta = json.loads((root / 'eval_final.json').read_text())[key]
    print(f"\n{name} run={item['run_id']}")
    print(json.dumps(meta, indent=1))
    video = root / 'eval_video.mp4'
    if video.exists(): display(Video(str(video), embed=True, width=640))
